# 평가셋 추론 (H7, 준아)

검증셋에서 정해 둔 모델·문턱·구간을 그대로 평가 몫에 적용해 예측 파일 다섯 개를 낸다. 여기서는 아무것도 새로 정하지 않는다.

| 칸 | 모델 | 쓰는 것 | 문턱(검증셋에서 정함) | 나오는 파일 |
|---|---|---|---|---|
| 5 | B1: OCR 글 + v9n | 저장소 `package/model_v9n` | 0.922496 | `pred_b1_test.csv` |
| 6 | B1+ TF-IDF | `models/model_b1plus_tfidf.pkl` | pkl 안의 값(0.296403) | `pred_b1plus_tfidf_test.csv` |
| 7 | B1+ KLUE (seed 0) | `models/ckpt_s0/` | 0.953839 | `pred_b1plus_klue_test.csv`, 글 임베딩 `emb_b1plus_klue_test.npy` |
| 8 | M2 (다영) | 칸 7 의 글 임베딩 + `models/m2/` | 0.657093 | `pred_m2_test.csv` |
| 9 | M3 | 칸 8 의 M2 점수 + Claude Opus 5.5 | 구간 [0.05, 0.8) | `pred_m3_test.csv` |

칸 7 → 8 → 9 는 앞 칸의 출력을 받으므로 순서대로 돌린다. 각 칸은 먼저 **검증 몫을 다시 돌려 채현·다영이 낸 값과 같은지** 확인한 뒤 평가 몫 파일을 쓴다. 다르면 멈춘다.

**준비 (한 번만)**
1. Google Drive `내 드라이브/AIX_H7/` 에 채현이 준 `models/`·`check/` 를 올린다.
2. 같은 폴더에 `test/` 를 만들고 준아가 가진 것을 넣는다: `split_v1.csv`(학습·검증·평가 전부), `meta.csv`, `ocr.csv`, 이미지. 이미지는 `test/` 아래에서 meta 의 `file` 열 경로로 찾는다(예: `test/images/rimg_....jpg`).
3. Colab 보안 비밀(왼쪽 열쇠)에 `GITHUB_TOKEN`(저장소 읽기 권한)과 `ANTHROPIC_API_KEY`(채현이 준 전용 키, 사용 한도 걸림)를 넣고 노트북 액세스를 켠다. 키를 셀에 직접 적지 않는다.
4. 런타임 > 런타임 유형 변경 > T4 GPU. 그다음 위에서 아래로 차례로 실행한다.

**출력에 대해:** 이 노트북은 평가 몫의 라벨(양성 수, unsure 수)을 출력하지 않는다. 채현은 평가 몫 라벨·이미지·OCR 을 보지 않기로 했으므로, 채현에게는 **칸 10 출력만** 보낸다.

In [ ]:
# 1. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Drive 연결, 동결 커밋의 코드 받기
FREEZE_COMMIT = '<11.16 동결 커밋>'   # 채현이 H7 때 적어 준다. 이 커밋으로 못 바꾸면 멈춘다
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, shutil, subprocess, json, glob
H7 = '/content/drive/MyDrive/AIX_H7'
T, M, CHK, OUT = f'{H7}/test', f'{H7}/models', f'{H7}/check', f'{H7}/runs_test'
for p in [f'{T}/split_v1.csv', f'{T}/meta.csv', f'{T}/ocr.csv', f'{M}/model_b1plus_tfidf.pkl', f'{M}/ckpt_s0', CHK]:
    assert os.path.exists(p), f'{p} 가 없다. 준비 1·2 를 확인'
tok = userdata.get('GITHUB_TOKEN')
if os.path.isdir('/content/AI_X'):
    shutil.rmtree('/content/AI_X')
r = subprocess.run(['git', 'clone', '-q', '-b', 'image-spec', f'https://{tok}@github.com/chlee1027/AI_X.git', '/content/AI_X'], capture_output=True, text=True)
assert r.returncode == 0, r.stderr.replace(tok, '***')
del tok
r = subprocess.run(['git', '-C', '/content/AI_X', 'checkout', '-q', FREEZE_COMMIT], capture_output=True, text=True)
assert r.returncode == 0, f'동결 커밋 {FREEZE_COMMIT} 로 못 바꿨다: {r.stderr}'
os.chdir('/content/AI_X/image_models')
os.makedirs(OUT, exist_ok=True)
!git log --oneline -1

In [ ]:
# 3. 패키지 (pkl 을 만든 판에 맞춘다: scikit-learn 1.5.1, transformers 4.46.3, anthropic 2.8.0)
!pip -q install scikit-learn==1.5.1 transformers==4.46.3 anthropic==2.8.0
import sklearn, transformers, anthropic, torch, pandas as pd, numpy as np
print('sklearn', sklearn.__version__, '| transformers', transformers.__version__, '| anthropic', anthropic.__version__,
      '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert sklearn.__version__ == '1.5.1', 'scikit-learn 판이 다르다. 런타임 > 세션 다시 시작 뒤 칸 2 부터 다시'

In [ ]:
# 4. 데이터 확인 (평가 몫은 장 수만 본다)
sp = pd.read_csv(f'{T}/split_v1.csv', dtype=str, keep_default_na=False)
meta = pd.read_csv(f'{T}/meta.csv', dtype=str, keep_default_na=False)
ocr = pd.read_csv(f'{T}/ocr.csv', dtype=str, keep_default_na=False)
print('split:', sp['split'].value_counts().to_dict())
test_ids = set(sp.loc[sp['split'] == 'test', 'image_id'])
assert test_ids, 'split_v1.csv 에 test 행이 없다'
assert test_ids <= set(meta['image_id']), 'meta 에 없는 test 이미지가 있다'
print('OCR 행이 없는 test 이미지:', len(test_ids - set(ocr['image_id'])), '장 (빈 글자로 둔다)')
files = meta.loc[meta['image_id'].isin(test_ids), 'file']
missing = [f for f in files if not os.path.isfile(f'{T}/{f}')]
print('파일이 없는 test 이미지:', len(missing), '장', missing[:3])
assert not missing, 'meta 의 file 열 경로로 test/ 아래 이미지를 찾지 못했다'
A = f'--split {T}/split_v1.csv --meta {T}/meta.csv --ocr {T}/ocr.csv --allow-test --out {OUT}'

def check_test(path):
    """예측 파일이 평가 몫 전부를 한 번씩 덮는지."""
    d = pd.read_csv(path, dtype={'image_id': str})
    assert d['image_id'].is_unique and set(d['image_id']) == test_ids, f'{path}: image_id 가 평가 몫과 다르다'
    assert d['score'].notna().all(), f'{path}: 빈 점수'
    print(f'  {os.path.basename(path)}: {len(d)}장, pred=1 {int(d["pred"].sum())}장')

def check_val(mine, ref, tol=1e-4):
    """검증 몫 재현: 여기서 다시 낸 점수와 채현·다영이 낸 점수의 최대 차이."""
    a = pd.read_csv(mine, dtype={'image_id': str}); b = pd.read_csv(ref, dtype={'image_id': str})
    m = b.merge(a, on='image_id', suffixes=('_ref', ''))
    assert len(m) == len(b), f'{ref} 의 image_id 가 다시 낸 파일에 다 있지 않다'
    diff = float(np.abs(m['score_ref'] - m['score']).max())
    print(f'  검증 재현 {os.path.basename(ref)}: {len(m)}장, 최대 차이 {diff:.2e}')
    assert diff <= tol, '검증 몫 점수가 다시 나오지 않는다. 멈추고 채현에게 알린다'

In [ ]:
# 5. B1: OCR 글 + v9n (학습 없음). 문턱 0.922496 = 검증셋 F1 최대 (팀 비교표의 B1 값과 같은 문턱)
!python b1_v9n.py $A --config configs/b1.yaml --score-only --thr 0.922496
check_val(f'{OUT}/b1/pred_b1_val.csv', f'{CHK}/pred_b1_val.csv')
check_test(f'{OUT}/b1/pred_b1_test.csv')

In [ ]:
# 6. B1+ TF-IDF: 저장한 모델로 점수만 낸다. 문턱은 pkl 안의 검증셋 값
!python b1plus_tfidf.py $A --config configs/b1plus_tfidf.yaml --infer-only {M}/model_b1plus_tfidf.pkl
check_val(f'{OUT}/b1plus_tfidf/pred_b1plus_tfidf_val.csv', f'{CHK}/pred_b1plus_tfidf_val.csv')
check_test(f'{OUT}/b1plus_tfidf/pred_b1plus_tfidf_test.csv')

In [ ]:
# 7. B1+ KLUE seed 0 체크포인트: 점수와 글 임베딩(768차원). 문턱 0.953839 = seed 0 검증셋 F1 최대
import time; t0 = time.time()
!python b1plus_klue.py $A --config configs/b1plus_klue.yaml --infer-only {M}/ckpt_s0 --thr 0.953839
print(f'걸린 시간 {(time.time() - t0) / 60:.1f}분')
K = f'{OUT}/b1plus_klue'
# 검증 몫 재현: 다영에게 넘긴 H6 검증 임베딩과 같은지 (float16 이라 1e-2 안이면 같은 것으로 본다)
e_new = np.load(f'{K}/emb_b1plus_klue_val.npy').astype(np.float32)
i_new = pd.read_csv(f'{K}/emb_b1plus_klue_val_ids.csv', dtype={'image_id': str})
e_ref = np.load(f'{CHK}/emb_b1plus_klue_val.npy').astype(np.float32)
i_ref = pd.read_csv(f'{CHK}/emb_b1plus_klue_val_ids.csv', dtype={'image_id': str})
pos = dict(zip(i_new['image_id'], i_new['row']))
assert set(i_ref['image_id']) <= set(pos), 'H6 검증 임베딩의 image_id 가 다시 낸 것에 다 있지 않다'
diff = float(np.abs(e_new[[pos[i] for i in i_ref['image_id']]] - e_ref[i_ref['row'].to_numpy()]).max())
print(f'  검증 재현 H6 임베딩: {len(i_ref)}장, 최대 차이 {diff:.2e}')
assert diff <= 1e-2, '검증 몫 임베딩이 다시 나오지 않는다. 멈추고 채현에게 알린다'
check_test(f'{K}/pred_b1plus_klue_test.csv')
e_t = np.load(f'{K}/emb_b1plus_klue_test.npy')
i_t = pd.read_csv(f'{K}/emb_b1plus_klue_test_ids.csv', dtype={'image_id': str})
assert set(i_t['image_id']) == test_ids and not np.isnan(e_t.astype(np.float32)).any()
print('  test 글 임베딩', e_t.shape, e_t.dtype)

In [ ]:
# 8. M2 (다영): 이미지 512 + 칸 7 글 임베딩 768 -> 로지스틱 회귀 시드 0·1·2 평균. 문턱 0.657093 = 검증셋 F1 최대
# 다영 추론 코드는 models/m2/ 에 들어온다. 부르는 방식(아래 인자)은 다영 코드를 받은 뒤 11.16 H7 전에 맞춘다.
M2 = f'{M}/m2'
assert os.path.isfile(f'{M2}/infer_m2.py'), 'models/m2/infer_m2.py 가 없다. 다영 M2 추론 코드를 받아 넣는다'
for which in ['val', 'test']:
    !python {M2}/infer_m2.py --split {T}/split_v1.csv --meta {T}/meta.csv --images {T} --which {which} --text-emb {K}/emb_b1plus_klue_{which}.npy --text-emb-ids {K}/emb_b1plus_klue_{which}_ids.csv --model-dir {M2} --out {OUT}/m2/m2_raw_{which}.csv
check_val(f'{OUT}/m2/m2_raw_val.csv', f'{CHK}/M2_val_scores.csv')
import imgcommon as C
raw = pd.read_csv(f'{OUT}/m2/m2_raw_test.csv', dtype={'image_id': str})
C.write_pred(f'{OUT}/m2/pred_m2_test.csv', raw['image_id'], raw['score'], 0.657093190891922, 'm2', 'dayoung', FREEZE_COMMIT)
check_test(f'{OUT}/m2/pred_m2_test.csv')

In [ ]:
# 9. M3: M2 점수가 [0.05, 0.8) 인 이미지만 Claude Opus 5.5 에 묻는다. 구간 밖은 M2 그대로
ZONE = (0.05, 0.8)
MAX_CALLS = 400          # 이보다 많으면 멈춘다(검증셋에서는 이미지의 2.27% 가 구간에 들었다)
base = pd.read_csv(f'{OUT}/m2/pred_m2_test.csv')
n = int(((base['score'] >= ZONE[0]) & (base['score'] < ZONE[1])).sum())
print(f'구간에 든 test 이미지 {n}장 = 호출 수, 어림 비용 ${n * 0.0123:.2f} (검증셋 실측 장당 $0.0123)')
assert n <= MAX_CALLS, '호출 수가 예상보다 많다. 멈추고 채현에게 알린다'
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
!python m3_claude.py $A --config configs/m3.yaml --base {OUT}/m2/pred_m2_test.csv --images {T} --zone {ZONE[0]} {ZONE[1]}
del os.environ['ANTHROPIC_API_KEY']
check_test(f'{OUT}/m3/pred_m3_test.csv')
# 끊겨서 다시 돌리면 runs_test/m3/claude_cache.jsonl 에 저장된 답은 다시 묻지 않는다

In [ ]:
# 10. 요약 (채현에게는 이 칸 출력만 보낸다. 라벨은 없다)
print('commit', FREEZE_COMMIT)
for f in sorted(glob.glob(f'{OUT}/*/pred_*_test.csv')):
    check_test(f)
rep = json.load(open(f'{OUT}/m3/report_m3_test.json', encoding='utf-8'))
print('M3:', {k: rep[k] for k in ('zone', 'calls', 'answered', 'errors', 'cost_usd', 'served', 'prompt_hash', 'config_hash', 'commit')})

## 끝나면

1. 칸 10 출력만 채현에게 보낸다. 예측 파일(`runs_test/*/pred_*_test.csv`)은 준아가 `score.py` 로 채점한다.
2. Colab 보안 비밀에서 `ANTHROPIC_API_KEY` 를 지운다. 채현이 11.17 채점 뒤 키를 폐기한다.
3. 한 번 돌린 뒤에는 모델·문턱·구간을 바꿔 다시 돌리지 않는다(11.16 동결).